In [1]:
import sys
import os

# Add the project root directory to Python's path
# This allows us to import 'qgcn_lib' from inside 'examples/notebooks'
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)



In [2]:
import torch
import os
import time
import math
import tqdm
from torch_geometric.nn import DeepGraphInfomax

# --- 1. Import from our Library ---
from qgcn_lib.datasets import ExperimentDataset 
from qgcn_lib.nn import SummaryMLP, QGCNConv
from qgcn_lib.utils import set_all_seeds, feature_shuffling_corruption, extract_experiment_subgraph

In [3]:
dataset_name = 'cora' 

# Construct the path relative to this script
# We assume data is in a './data' folder next to this script
current_dir = os.getcwd()
file_path = os.path.join(current_dir, '..', 'data', f'{dataset_name}.pt')

print(f"--> Initializing Library Wrapper for: {dataset_name}")

# USE THE WRAPPER: This creates a PyG-compatible dataset object
try:
    dataset = ExperimentDataset(root=os.path.join(current_dir, 'data'), file_path=file_path)
    data = dataset[0]  # Get the graph object
    

    
    print(f"--> Data Loaded Successfully via qgcn_lib")
    print(f"    Nodes: {data.num_nodes} | Features: {data.num_features}")
    
except FileNotFoundError:
    print(f"\n[ERROR] Could not find {dataset_name}.pt")
    print(f"Please ensure you have placed the file at: {file_path}")
    exit()


--> Initializing Library Wrapper for: cora
--> Data Loaded Successfully via qgcn_lib
    Nodes: 2708 | Features: 1433


In [4]:
data

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [13]:
def train_iterative_chunking(
    data, 
    num_chunks=10, 
    edges_per_chunk=1000, 
    iters_per_chunk=10, 
    target_dim=32, # WE HAVE NEVER USED DIM REDUCTIOAN ANYWHERE IN OUR EXPERIMENTS. THIS IS JUST FOR TUTORIAL PURPOSES. 
    save_dir="./checkpoints"
):
    """
    Executes chunk-based training for QGCN to manage memory footprint on large graphs.
    Saves a discrete model checkpoint after processing each chunk.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Starting chunked training on: {device}")
    
    os.makedirs(save_dir, exist_ok=True)
    
    num_total_nodes = data.x.size(0)
    hidden_channels = math.ceil(math.log2(target_dim))
    
    model = None
    optimizer = None

    for chunk_idx in range(num_chunks):
        print(f"\n--- Processing Chunk {chunk_idx + 1}/{num_chunks} ---")
        
        # 1. Subgraph Extraction
        t_extract = time.time()
        sub_data = extract_experiment_subgraph(
            data, 
            num_edges=edges_per_chunk, 
            edge_chunk_idx=chunk_idx, 
            target_dim=target_dim
        )
        features = sub_data.x.to(device)
        edge_index = sub_data.edge_index.to(device)
        print(f"Extraction & PCA time: {time.time() - t_extract:.2f}s")

        # 2. Lazy Initialization (Triggers only on the first chunk)
        if model is None:
            set_all_seeds(123)
            
            encoder = QGCNConv(
                in_channels=features.size(1), 
                points=num_total_nodes, 
                hidden_channels=hidden_channels, 
                q_depth=5
            )
            summary = SummaryMLP(hidden_channels)
            
            model = DeepGraphInfomax(
                hidden_channels=hidden_channels,
                encoder=encoder,
                summary=summary,
                corruption=feature_shuffling_corruption
            ).to(device)
            
            
            optimizer = torch.optim.Adam([
                {'params': model.encoder.qc.parameters(), 'lr': 0.01},
                {'params': model.encoder.local_mp.parameters(), 'lr': 0.01},
                {'params': model.summary.parameters(), 'lr': 0.001},
                {'params': model.encoder.q_proj.parameters(), 'lr': 0.001},
                {'params': model.encoder.prelu.parameters(), 'lr': 0.001},
                {'params': [model.encoder.bias], 'lr': 0.001}
            ])

        # 3. Chunk Training Loop
        model.train()
        chunk_loss = 0.0
        
        for iteration in range(iters_per_chunk):
            t_iter = time.time()
            
            optimizer.zero_grad()
            pos_z, neg_z, summary_vec = model(features, edge_index)
            loss = model.loss(pos_z, neg_z, summary_vec)
            loss.backward()
            optimizer.step()
            
            iter_time = time.time() - t_iter
            chunk_loss += loss.item()
            
            print(f"  Iter {iteration + 1:02d}/{iters_per_chunk} | Loss: {loss.item():.4f} | Time: {iter_time:.2f}s")

        # 4. State Checkpointing
        avg_loss = chunk_loss / iters_per_chunk
        print(f"Chunk {chunk_idx + 1} complete. Average Loss: {avg_loss:.4f}")
        
        t_save = time.time()
        save_path = os.path.join(save_dir, f"qgcn_checkpoint_chunk_{chunk_idx}.pt")
        
        checkpoint = {
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
        }
        torch.save(checkpoint, save_path)
        # Save only the state_dict to minimize I/O overhead
        torch.save(model.state_dict(), save_path)
        print(f"Checkpoint saved to {save_path} in {time.time() - t_save:.2f}s")

    print("\nTraining complete.")
    return model



In [14]:
# USAGE EXAMPLE:
# TO REPEATE SIMILAR EXPERIMENTS AS THE PAPER, ONE NEED TO PASS ARGUMENTS IN THE WAY THAT FULL GRAPH TOPOLOGY AND FEATURES ARE DISCOVERED. BELOW IS JUST AN EXAMPLE ON THE SMALL SCALE CORA.
final_model = train_iterative_chunking(data, num_chunks=1, edges_per_chunk=1000, iters_per_chunk=10)

Starting chunked training on: cpu

--- Processing Chunk 1/1 ---
--> [Mode] Edge-Centric: Chunk 0 (1000 edges per chunk)
--> [PCA] Applying PCA: 1433 -> 32 dimensions.
--> Resulting Graph: 2708 Nodes | 1000 Edges | 32 Features

Extraction & PCA time: 0.04s
  Iter 01/10 | Loss: 1.3899 | Time: 18.55s
  Iter 02/10 | Loss: 1.3890 | Time: 17.97s
  Iter 03/10 | Loss: 1.3891 | Time: 17.93s
  Iter 04/10 | Loss: 1.3896 | Time: 18.07s
  Iter 05/10 | Loss: 1.3884 | Time: 18.10s
  Iter 06/10 | Loss: 1.3886 | Time: 17.83s
  Iter 07/10 | Loss: 1.3890 | Time: 18.15s
  Iter 08/10 | Loss: 1.3885 | Time: 18.44s
  Iter 09/10 | Loss: 1.3884 | Time: 18.34s
  Iter 10/10 | Loss: 1.3873 | Time: 18.56s
Chunk 1 complete. Average Loss: 1.3888
Checkpoint saved to ./checkpoints/qgcn_checkpoint_chunk_0.pt in 0.00s

Training complete.


In [15]:
def extract_encodings(model, data, target_dim=32):
    """
    Passes data through the trained QGCN model to get the quantum embeddings.
    """
    # 1. Set model to evaluation mode 
    model.eval()
    
    # Ensure we are using the same device as the model
    device = next(model.parameters()).device
    print(f"--> Extracting encodings on: {device}")

    # 2. Prepare the data (Apply the exact same PCA reduction used in training)
    # We leave num_nodes and num_edges as None so it processes the graph you pass it
    print(f"--> Formatting features to match model's expected input dimension ({target_dim})...")
    eval_data = extract_experiment_subgraph(data, target_dim=target_dim)
    
    features = eval_data.x.to(device)
    edge_index = eval_data.edge_index.to(device)

    # 3. Forward Pass 
    print("--> Running quantum message passing...")
    with torch.no_grad():
        # DGI returns (real_embeddings, corrupted_embeddings, summary)
        # We only want the real embeddings (z)
        z, _, _ = model(features, edge_index)
        
    print(f"✅ Successfully extracted embeddings. Shape: {z.shape}")
    return z

# --- How to use it ---
# Assuming 'final_model' is the model returned by your training loop
# and 'data' is the graph you want to evaluate:

# z_embeddings = extract_encodings(final_model, data, target_dim=32)

# Save them immediately so you don't have to compute them again!
# torch.save(z_embeddings.cpu(), "final_cora_embeddings.pt")

In [16]:
z = extract_encodings(final_model, data, target_dim=32)

--> Extracting encodings on: cpu
--> Formatting features to match model's expected input dimension (32)...
--> [Mode] Full Graph: No node or edge limits applied.
--> [PCA] Applying PCA: 1433 -> 32 dimensions.
--> Resulting Graph: 2708 Nodes | 10556 Edges | 32 Features

--> Running quantum message passing...
✅ Successfully extracted embeddings. Shape: torch.Size([2708, 5])


In [17]:
from qgcn_lib.utils import perform_kmeans_clustering



# 1. Cluster Analysis (K-Means)
# We ask K-Means to find 7 clusters in the 5-dimensional quantum latent space
labels, z_np, score = perform_kmeans_clustering(z, 7)



Silhouette Score (k=7): 0.1885
